> **Provenance.** This notebook is from Wen & Wang (2025), *Code for: Seismic wave propagation simulation with quantum computing*, Zenodo, [doi:10.5281/zenodo.14868887](https://doi.org/10.5281/zenodo.14868887), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes in this repository: outputs cleared, and this note plus the setup cell below added. The notebook uses the classical simulated-annealing sampler by default; the D-Wave QPU sampler is left commented out, as in the original.

In [ ]:
# Added for this repository: make the top-level `core` package importable
# when the notebook is run from the notebooks/ folder.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
import sys
import numpy as np
from core import stagger_FD_coeff as sfci
from core.utils import Logger
from neal import SimulatedAnnealingSampler
import time
from core.QUBO_solvers import IterSampleSolver
import matplotlib.pyplot as plt
from contextlib import redirect_stdout
from tqdm import tqdm

pi = np.pi

In [ ]:
v = 4500
h = 10
tao = 0.001
freq_max_set = [174]
M_set = [8]
sample_num = 100
max_iter_num = 20
max_iter4_solver = 5
epsilon4linsys = 1e-20
sampler = SimulatedAnnealingSampler()


# Influence of Regularization

In [ ]:
# this program will take a long time !!!
log_filename = f"Inf_reg_{time.strftime('%Y%m%d%H%M', time.localtime())}.txt"
with open(log_filename, "w") as log_file, redirect_stdout(
    Logger(log_filename, sys.stdout)
):

    print(f"v={v}")
    print(f"freq={174}")
    print(f"M={8}")
    print(f"sample time = {sample_num}")
    print(f"max_iter_num = {max_iter_num}")
    print(f"max_iter_num_in_iterative_solver = {max_iter4_solver}")
    print("b_alpha = 1*0.1*it change in big loop")  # alpha = 1 * 0.8 ** it
    print("s_alpha = b_alpha change in small loop")

    for it_num in tqdm(range(20)):
        print(f" >>> Start the {it_num} experiment.")
        qa_a0_nonreg_dict = {}
        for M, freq_max in zip(M_set, freq_max_set):
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                R=10,
                fixed_point=1,
                epsilon=1e-10,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_energy",
            )
            qa_a0 = linear_sys.solve(
                M=M,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                solver=sis.solve,
                epsilon=epsilon4linsys,
                show_process=False,
            )
            qa_a0_nonreg_dict[f"Non_reg"] = qa_a0

        qa_a0_reg_dict = {}
        for M, freq_max in zip(M_set, freq_max_set):
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                R=10,
                fixed_point=1,
                epsilon=1e-10,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_energy",
            )
            qa_a0 = linear_sys.solve(
                M=M,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                epsilon=epsilon4linsys,
                solver=sis.solve,
                show_process=False,
                SampleSolver_param_dict={"alpha" : 1.0, "drate" : 0.1}
            )
            qa_a0_reg_dict[f"Reg"] = qa_a0

        print(f" <<< End the {it_num} experiment.")

# Comparation of new and old QA algorithm for solving linear system of equations

In [ ]:
log_filename = f"new_old_{time.strftime('%Y%m%d%H%M', time.localtime())}.txt"
with open(log_filename, "w") as log_file, redirect_stdout(
    Logger(log_filename, sys.stdout)
):

    print(f"v={v}")
    print(f"freq={174}")
    print(f"M={8}")
    print(f"sample time = {sample_num}")
    print(f"max_iter_num = {max_iter_num}")
    print(f"max_iter_num_in_iterative_solver = {max_iter4_solver}")
    print(f"epsilon = {epsilon4linsys}")
    print("non-regularization!")
    print("L varies between 0.0001 and 100")
    step_L_list = np.linspace(0.0001, 100, 51)

    for it_num in tqdm(range(20)):
        print(f" >>> Start the {it_num} experiment.")
        qa_a0_nonreg_dict = {}
        for M, freq_max in zip(M_set, freq_max_set):
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                R=10,
                fixed_point=1,
                epsilon=epsilon4linsys,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_energy",
                regularization=False,
                # num_spin_reversal_transforms=10,
                # postprocess=True
            )
            qa_a0 = linear_sys.solve(
                M=M,
                alpha=0,
                beta=1,
                epsilon=epsilon4linsys,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                solver=sis.solve,
                show_process=False,
            )
            qa_a0_nonreg_dict[f"New method"] = qa_a0

        qa_a0_reg_dict = {}
        step_L = step_L_list[it_num]
        for M, freq_max in zip(M_set, freq_max_set):
            print(f"!!!L={step_L}!!!")
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                L=step_L,
                # L and steplen are hard to estimate, but can make significant influence to result!
                steplen=0.75,
                R=10,
                fixed_point=1,  # in original algorithm fp must equal to 1, because x\in [0,2]
                epsilon=epsilon4linsys,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_energy",
                regularization=False,  # alpha=1*0.1**it Which is same as reg-lstsq method
                codingMod="p",
                # num_spin_reversal_transforms=10,
                # postprocess=True
            )
            qa_a0 = linear_sys.solve(
                M=M,
                # alpha=0,
                # beta=1,
                epsilon=epsilon4linsys,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                solver=sis.original_QUBO_solve,
                show_process=False,
            )
            qa_a0_reg_dict["Souza's Method"] = qa_a0

        print(f" <<< End the {it_num} experiment.")

# Influence of adding non-linear object function

In [ ]:
log_filename = f"add_nonlinear_{time.strftime('%Y%m%d%H%M', time.localtime())}.txt"
with open(log_filename, "w") as log_file, redirect_stdout(
    Logger(log_filename, sys.stdout)
):

    print(f"v={v}")
    print(f"freq={174}")
    print(f"M={8}")
    print(f"sample time = {sample_num}")
    print(f"max_iter_num = {max_iter_num}")
    print(f"max_iter_num_in_iterative_solver = {max_iter4_solver}")
    print("b_alpha = 1*0.1*it change in big loop")  # alpha = 1 * 0.8 ** it
    print("s_alpha = b_alpha change in small loop")

    sampler = SimulatedAnnealingSampler()
    for it_num in tqdm(range(20)):
        print(f" >>> Start the {it_num} experiment.")
        qa_a0_nonreg_dict = {}
        for M, freq_max in zip(M_set, freq_max_set):
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                R=10,
                fixed_point=1,
                epsilon=1e-5,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_energy",
                regularization=False,
                # num_spin_reversal_transforms=10,
                # postprocess=True
            )
            qa_a0 = linear_sys.solve(
                M=M,
                # beta=1,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                solver=sis.solve,
                epsilon=epsilon4linsys,
                show_process=False,
            )
            qa_a0_nonreg_dict[f"original"] = qa_a0

        qa_a0_reg_dict = {}
        for M, freq_max in zip(M_set, freq_max_set):
            linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
            sis = IterSampleSolver(
                sampler=sampler,
                x0=np.ones(M) / 100,  # initial vector of Delta a
                num_sample=sample_num,
                R=10,
                fixed_point=1,
                epsilon=1e-5,
                maxiter=max_iter4_solver,  # max iteration in small iteration
                resultMod="lowest_obj",
                regularization=False,  # alpha=1*0.1**it Which is same as reg-lstsq method
                # num_spin_reversal_transforms=10,
                # postprocess=True
            )
            qa_a0 = linear_sys.solve(
                M=M,
                # beta=1,
                max_iter_num=max_iter_num,
                mode_of_construct="direct",
                epsilon=epsilon4linsys,
                solver=sis.solve,
                show_process=False,
                add_non_linear_obj=True,
            )
            qa_a0_reg_dict[f"add_non_linear_obj"] = qa_a0

        print(f" <<< End the {it_num} experiment.")

# Plot the Result

In [ ]:
label_set = ["Non-reg", "Reg"]
max_iter = max_iter_num
epsilon = 1e-5

log_file = open("Inf_reg_202406171657.txt", "r")
lf = log_file.readlines()
log_file.close()

data_list = []
counter = 0
for line in lf:
    if "Current object function" in line and "*" not in line:
        content = line.strip().split()
        data_list.append(float(content[-1]))
        if counter == 0:
            original_obj = float(content[-1])
        counter += 1

indx = np.argwhere(
    (original_obj == np.array(data_list))
).squeeze()  # index of original object function
indx_non_reg = indx[0::2]
indx_reg = indx[1::2]

nreg_obj_data = {}
reg_obj_data = {}
for ind in range(len(indx_non_reg)):
    nreg_obj_data[ind] = np.array(
        [data for data in data_list[indx_non_reg[ind] + 1 : indx_reg[ind]]]
    )
    if ind < len(indx_non_reg) - 1:
        reg_obj_data[ind] = np.array(
            [data for data in data_list[indx_reg[ind] + 1 : indx_non_reg[ind + 1]]]
        )
    else:
        reg_obj_data[ind] = np.array([data for data in data_list[indx_reg[ind] + 1 :]])


# 计算中位数曲线
def median_curve_and_error(obj_data, max_iter_num):
    obj_in_each_iter = [[]] * max_iter_num  
    for j, obj_line in enumerate(list(obj_data.values())):
        if (
            len(obj_line) != max_iter_num
        ):  
            obj_line = np.hstack(
                [obj_line, [obj_line[-1]] * (max_iter_num - len(obj_line))]
            )
        for i in range(len(obj_line)):
            obj_in_each_iter[i] = obj_in_each_iter[i] + [obj_line[i]]

    median_line = np.zeros(max_iter_num)
    # RMSE = np.zeros(20) 
    error_lines = np.zeros([2, max_iter_num])  

    for i, obj_list in enumerate(obj_in_each_iter):
        if len(obj_list) != 0:
            median_line[i] = np.median(obj_list)  # np.array(obj_list).mean()  #
            # RMSE[i] = np.sum(np.abs(mean_line_nreg[i] - np.array(obj_list)))/len(obj_list)
            error_lines[0, i] = np.abs(
                np.min(median_line[i] - np.array(obj_list))
            )  
            error_lines[1, i] = -np.abs(
                np.max(median_line[i] - np.array(obj_list))
            )  
    return median_line, error_lines


median_line_nreg, error_lines_nreg = median_curve_and_error(nreg_obj_data, max_iter)
median_line_reg, error_lines_reg = median_curve_and_error(reg_obj_data, max_iter)

lw = 1.1
fig, ax = plt.subplots(1, 1, figsize=(5, 7))
ax.spines["left"].set_linewidth(lw)
ax.spines["top"].set_linewidth(lw)
ax.spines["right"].set_linewidth(lw)
ax.spines["bottom"].set_linewidth(lw)

for ind in range(len(indx_non_reg)):
    ax.plot(
        [i for i in range(1, len(nreg_obj_data[ind]) + 1)],
        nreg_obj_data[ind],
        "o-",
        markersize=5,
        color="crimson",
        alpha=0.3,
        linewidth=0.5,
    )
    ax.plot(
        [i for i in range(1, len(reg_obj_data[ind]) + 1)],
        reg_obj_data[ind],
        "o-",
        markersize=5,
        color="skyblue",
        alpha=0.3,
        linewidth=0.5,
    )

ax.plot(
    [i for i in range(1, len(median_line_nreg) + 1)],
    median_line_nreg,
    "o-",
    color="firebrick",
    markeredgecolor="maroon",
    markerfacecolor="crimson",
    label=label_set[0],
)

ax.fill_between(
    [i for i in range(1, len(median_line_nreg) + 1)],
    error_lines_nreg[0] + median_line_nreg,
    error_lines_nreg[1] + median_line_nreg,
    color="crimson",
    alpha=0.2,
)

# ax.errorbar([i for i in range(1, len(mean_line_nreg)+1)],mean_line_nreg,yerr=RMSE_nreg,fmt='-o',color = 'black',label="0.5-2ndHeating",elinewidth=1,capsize=3)
ax.plot(
    [i for i in range(1, len(median_line_reg) + 1)],
    median_line_reg,
    "o-",
    color="darkblue",
    markeredgecolor="darkblue",
    markerfacecolor="royalblue",
    label=label_set[1],
)

ax.fill_between(
    [i for i in range(1, len(median_line_reg) + 1)],
    error_lines_reg[0] + median_line_reg,
    error_lines_reg[1] + median_line_reg,
    color="skyblue",
    alpha=0.2,
)

ax.set_ylabel(
    "Object Function",
    fontfamily="Times New Roman",
    fontsize=13,
    fontweight="bold",
    labelpad=-5,
)
ax.set_xlabel(
    "Iterations", fontfamily="Times New Roman", fontsize=13, fontweight="bold"
)
ax.set_yscale("log")
ax.set_ylim([1e-10, 1e15])
ax.set_xlim([1, 20])
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_yticks([1e-10, 1e-5, 1, 1e5, 1e10, 1e15])
ax.tick_params(
    top=True,
    right=True,
    which="major",
    direction="in",
    length=4,
    width=1.0,
    colors="black",
    labelsize=10,
)

ax.tick_params(
    top=True,
    right=True,
    direction="in",
    which="minor",
    length=2,
    width=0.8,
    colors="black",
)

ax.grid(visible=True, axis="both", which="major", linestyle="--", linewidth=0.5)

ax.axhline(
    y=epsilon,
    linestyle="--",
    linewidth=1.2,
    color="black",
    label="$\epsilon=10^{-5}$",
)
lw = 1.5
ax.spines["left"].set_linewidth(lw)
ax.spines["top"].set_linewidth(lw)
ax.spines["right"].set_linewidth(lw)
ax.spines["bottom"].set_linewidth(lw)

ax.legend(
    title="",
    fontsize="small",
    loc="lower left",
    borderpad=1.0,  
    borderaxespad=2.0, 
    labelspacing=0.5,  
    edgecolor="black",
    handlelength=2.0,
)
# ax.legend(loc=1,fontsize='small')
# plt.tight_layout()
plt.show(block=True)
# fig.savefig("New_old_reuslt.pdf", dpi=600, format="pdf")